In [96]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [97]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [98]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [99]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download the `.pkg`
     - **Windows**: download the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   - In a terminal, run:
     ```bash
     ollama run llama3
     ```
   - This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. **Test the local server**
   - Run:
     ```bash
     curl http://localhost:11434
     ```
   - You should get a response like:
     ```json
     {"models": [...]}
     ```

4. **Use it from Python**
   - Install the client:
     ```bash
     pip install ollama
     ```
   - Example:
     ```python
     import ollama

     response = ollama.chat(
         model='llama3',
         messages=[{"role": "user", "content": your_prompt}]
     )

     print(response['message']['content'])
     ```


The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, so it can't try again with a corrected query.

We need something smarter. We need an agent.

In [100]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The FAQ doesn’t mention **Olama** specifically. If you mean running the course locally, the FAQ says you can do that instead of using Codespaces if you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

It also says that if you run locally, you should **document your setup** and keep the environment **reproducible**.

If you meant something else by “Olama,” let me know.


## Asking without tools

First, let's see what the LLM does without any tools. We ask it a
course-specific question and look at the answer.

In [101]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open and hasn’t started, you can often join after discovering it.\n\nA couple of things to check:\n- **Enrollment deadline**: Has registration closed?\n- **Course start date**: Is it already in progress?\n- **Seat availability / prerequisites**: Do you meet any requirements?\n- **Instructor or admin approval**: Some courses allow late enrollment only with permission.\n\nIf you want, I can help you draft a short message asking the course organizer whether you can still join.'

In [65]:
index.search('how to run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [102]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our
Python code, only a schema describing what the function does and what
arguments it takes. LLMs are language agnostic. At the end we're just
making an HTTP call, so we describe the tool in JSON rather than in
Python. The same schema would work from TypeScript or Java.

In [103]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

The `description` is the most important field, because the model reads
it to decide when to call the function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

In [104]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [105]:
len(response.output)

1

In [106]:
call = response.output[0]

In [107]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late? discovered the course just now enrollment late join"}', call_id='call_ndVwSt2UPmLWS1DaHgX4B3cC', name='search', type='function_call', id='fc_03c1905e129ad0a4006aadbfdbb0ec87d2b3bfcf54cfd41303', async_=None, caller=None, namespace=None, status='completed')

Look at the output. Instead of a message with the answer, the response
contains a `function_call` entry. The model decided it needs to search
the FAQ before answering. Rather than reply, it asked us to run the
search function first.

Look at the arguments too. The model didn't pass our question
verbatim. It judged the raw question wasn't the best query to search
with. So it rewrote our enrollment question into search keywords like
"enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our
`search` function, and serialize the result.

In [108]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course late? discovered the course just now enrollment late join'}

In [109]:
call.name

'search'

In [110]:
results = search(**args)

In [111]:
result_json = json.dumps(results, indent=2)

In [112]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [113]:
messages.append(call)

In [114]:
messages.append(function_call_output)

In [115]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late? discovered the course just now enrollment late join"}', call_id='call_ndVwSt2UPmLWS1DaHgX4B3cC', name='search', type='function_call', id='fc_03c1905e129ad0a4006aadbfdbb0ec87d2b3bfcf54cfd41303', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_ndVwSt2UPmLWS1DaHgX4B3cC',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certificate show t

Now we send this result back to the model.\
First, we add the model's output to the conversation history - the model needs to see its own\
function call.\
Then we add the tool result.

## Asking the model again

We call the API a second time with the expanded history:

In [116]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [117]:
print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, make sure you submit your project while submissions are still open.


We cannot simply make a nother call to LLM with this output, we have to include the history from the first call\
because LLMs are stateless in nature. 

This time the model has the original question, its own decision to
call `search`, and the FAQ results. It can now produce a proper
course-specific answer.

We have to send the whole history because LLMs are stateless between
API calls. The memory is the list you send as `input`. If you send
only the tool result, the model has no idea what's going on. So on
this second call we replay everything we have so far. That means the
question, the decision to call `search`, and the result we got back.

That's the full function-calling loop for a single turn. With plain
RAG we made one call, and here we make two. Turning RAG agentic means
more round-trips.

People call this pattern "agentic RAG", "tool use", or "function
calling". The idea behind all of them is the same. The LLM decides
which tools to call.

## Token usage and cost

We just made two API calls instead of one. Each call we send to the
model costs money, so it's worth checking how much one tool-using turn
actually costs.

The response has a `usage` field with the token counts:

In [118]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(788, 33)

For each model the provider publishes a price per million input tokens
and per million output tokens. Plug those numbers in to convert tokens
to dollars.

In [119]:
def calculate_gpt54mini_price(input_tokens, output_tokens): 
    INPUT_PRICE_PER_MILLION = 0.75  # prices for GPT-5.4-mini for 18th Dec 2026
    OUTPUT_PRICE_PER_MILLION = 4.5

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(786, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.000738


This usage is only for the second API call. The first call also has
its own usage and its own cost. That was the call where the model
decided to invoke `search`. Two calls means we pay twice. We pay even
more on the second call, because we resend the full history as input.

With a real agent loop the model can make many calls, so the costs add
up. Keep an eye on `usage` while you develop.

In [120]:
def calculate_agent_loop_cost(
    first_call_input,
    first_call_output,
    second_call_cached_input,
    second_call_new_input,
    second_call_output,
):
    """
    Calculate GPT-5.4 mini API cost for a two-call agent loop.

    Prices per 1M tokens:
        Input:        $0.75
        Cached input: $0.075
        Output:       $4.50
    """

    INPUT_PRICE = 0.75
    CACHED_INPUT_PRICE = 0.075
    OUTPUT_PRICE = 4.50

    # -------------------------
    # FIRST CALL
    # -------------------------

    first_input_cost = (
        first_call_input / 1_000_000
    ) * INPUT_PRICE

    first_output_cost = (
        first_call_output / 1_000_000
    ) * OUTPUT_PRICE

    first_call_total = (
        first_input_cost +
        first_output_cost
    )

    # -------------------------
    # SECOND CALL
    # -------------------------

    second_cached_input_cost = (
        second_call_cached_input / 1_000_000
    ) * CACHED_INPUT_PRICE

    second_new_input_cost = (
        second_call_new_input / 1_000_000
    ) * INPUT_PRICE

    second_output_cost = (
        second_call_output / 1_000_000
    ) * OUTPUT_PRICE

    second_call_total = (
        second_cached_input_cost +
        second_new_input_cost +
        second_output_cost
    )

    # -------------------------
    # TOTAL
    # -------------------------

    total_cost = (
        first_call_total +
        second_call_total
    )

    return {
        "first_call": {
            "input_cost": first_input_cost,
            "output_cost": first_output_cost,
            "total": first_call_total,
        },
        "second_call": {
            "cached_input_cost": second_cached_input_cost,
            "new_input_cost": second_new_input_cost,
            "output_cost": second_output_cost,
            "total": second_call_total,
        },
        "total_agent_cost": total_cost,
    }

In [80]:
result = calculate_agent_loop_cost(
    first_call_input=786,
    first_call_output=36,
    second_call_cached_input=786,
    second_call_new_input=2500,
    second_call_output=400,
)

print(f"First call:  ${result['first_call']['total']:.9f}")
print(f"Second call: ${result['second_call']['total']:.9f}")
print(f"Total:       ${result['total_agent_cost']:.9f}")

First call:  $0.000751500
Second call: $0.003733950
Total:       $0.004485450


## A developer prompt

So far we've relied on the model to figure out when to search. We make
that more reliable with a `developer` message that spells out how to
behave. This is where we give the agent its role. The same message
also pushes it toward multiple searches, so we get to watch the loop
run more than once.

In [121]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [122]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [123]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [87]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [124]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered course can I join"}
function_call: search {"query":"course registration open enrollment late join discovered course"}


In [125]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered course can I join"}', call_id='call_iJKEG5wxo53TM8Bb7SnbaHUu', name='search', type='function_call', id='fc_045be707bbe8836e006aadc13538f887d2880424810cb296e5', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course registration open enrollment late join discovered cours

In [126]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered late enroll registration can I join course"}
function_call: search {"query":"course late enrollment join discovered course can I join"}
function_call: search {"query":"enrollment open after course start join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still being accepted. You can also start learning from the materials now, even if you discovered the course late.

If you’d like, I can also help you figure out how to get started or what you need for the certificate.


In [127]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [128]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [129]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment discover course can I join"}
iteration #2...
function_call: search {"query":"course discovered late still join certificate live cohort project submission accepting submissions"}
iteration #3...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions, and the certificate is only available if you complete the course with a live cohort.

If you want, I can also explain the certificate requirements or how the capstone/project submission works.


In [130]:
result

'Yes — you can still join the course even if you just discovered it.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions, and the certificate is only available if you complete the course with a live cohort.\n\nIf you want, I can also explain the certificate requirements or how the capstone/project submission works.'

In [131]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess"}
iteration #2...
function_call: search {"query":"queen's gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so I’m not able to answer that from the course materials.

If you meant something specific in the course, try giving me the exact term or context from the lesson. Are there other areas you want to explore?
